# Unidad 8 · Comparaciones múltiples

Cuaderno de la base abierta de ingeniería (Análisis de Datos). Reproduce el ejemplo resuelto (capachos de
cosecha de arándanos) y la pieza del caso Estampa Sur (qué tintas difieren en el secado). Los datos están
en `../datos/`: el ejemplo en `u08_ejemplo.csv` (lo genera `verificacion/u08_comparaciones_multiples.py`) y
el caso en `secado.csv`.

In [ ]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

DATOS = Path("..") / "datos"


def pares_t(grupos):
    """Pruebas t de todos los pares con el CME del ANOVA (sin corregir). Devuelve [(a, b, diferencia, t, p)]."""
    k = len(grupos); N = sum(len(v) for v in grupos.values())
    cme = sum(((v - v.mean()) ** 2).sum() for v in grupos.values()) / (N - k)
    filas = []
    for a, b in combinations(grupos, 2):
        d = grupos[b].mean() - grupos[a].mean()
        ee = np.sqrt(cme * (1 / len(grupos[a]) + 1 / len(grupos[b])))
        t = d / ee
        filas.append((a, b, d, t, 2 * stats.t.sf(abs(t), N - k)))
    return filas, cme

## 1. Ejemplo · Tukey, Bonferroni y Dunnett

In [ ]:
df = pd.read_csv(DATOS / "u08_ejemplo.csv")
grupos = {g: v.kg_hora.to_numpy(float) for g, v in df.groupby("capacho", sort=False)}

tukey_ejemplo = pairwise_tukeyhsd(df.kg_hora, df.capacho, alpha=0.05)
print(tukey_ejemplo)

filas, cme = pares_t(grupos)
p_sin = [f[4] for f in filas]
bonferroni_ejemplo = multipletests(p_sin, method="bonferroni")[1]
for (a, b, d, t, p), pb in zip(filas, bonferroni_ejemplo):
    print(f"{b:>8} − {a:<8} d = {d:6.3f}  t = {t:6.3f}  p sin corregir = {p:.4f}  p Bonferroni = {pb:.4f}")

control = grupos["actual"]
otros = [g for g in grupos if g != "actual"]
dunnett_ejemplo = stats.dunnett(*[grupos[g] for g in otros], control=control, rng=np.random.default_rng(1))   # Dunnett usa integración numérica: semilla fija
print(dict(zip(otros, dunnett_ejemplo.pvalue.round(4))))
print(dunnett_ejemplo.confidence_interval(0.95))

## 2. Ejemplo · el HSD a mano

In [ ]:
k, n = len(grupos), 5
q = stats.studentized_range.ppf(0.95, k, len(df) - k)
hsd = q * np.sqrt(cme / n)
print(f"CME = {cme:.4f}  q = {q:.3f}  HSD = {hsd:.3f}")

## 3. Caso Estampa Sur · qué tintas difieren

In [ ]:
s = pd.read_csv(DATOS / "secado.csv")
tukey_caso = pairwise_tukeyhsd(s.secado_min, s.tinta, alpha=0.05)
print(tukey_caso)
grupos_caso = {g: v.secado_min.to_numpy(float) for g, v in s.groupby("tinta")}
filas_c, cme_c = pares_t(grupos_caso)
bonferroni_caso = multipletests([f[4] for f in filas_c], method="bonferroni")[1]
for (a, b, d, t, p), pb in zip(filas_c, bonferroni_caso):
    print(f"{b:>9} − {a:<9} d = {d:6.2f}  p sin corregir = {p:.4f}  p Bonferroni = {pb:.4f}")
fig = tukey_caso.plot_simultaneous()    # intervalos simultáneos de Tukey